# Facial Emotion Detection - Results Visualization\n\nThis notebook loads the training/evaluation results saved by `train_model.py` and visualizes the metrics, including K-Fold performance, confusion matrices, and ROC curves.

In [ ]:
import os\nimport json\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Update with your project root path if necessary\nproject_root = '../../'\nresults_dir = os.path.join(project_root, 'results/tables')\n\nplt.style.use('ggplot')\nsns.set_palette("deep")

## 1. Load Results Data

In [ ]:
def load_results():\n    all_results = {}\n    if not os.path.exists(results_dir):\n        print(f"Directory {results_dir} not found.")\n        return all_results\n        \n    for file in os.listdir(results_dir):\n        if file.endswith('.json'):\n            model_name = file.replace('results_', '').replace('kfold_results_', '').replace('.json', '')\n            with open(os.path.join(results_dir, file), 'r') as f:\n                data = json.load(f)\n            all_results[file] = {'model': model_name, 'data': data}\n    return all_results\n\nresults = load_results()\nprint(f"Loaded {len(results)} result files.")

## 2. Compare Model Performance

In [ ]:
def plot_model_comparison(results_dict, metric='accuracy'):\n    models = []\n    scores = []\n    errors = []\n    \n    for filename, res in results_dict.items():\n        data = res['data']\n        model_name = res['model']\n        \n        if 'n_folds' in data: # K-Fold result\n            if metric in data:\n                models.append(f"{model_name} (K-Fold)")\n                scores.append(data[metric]['mean'])\n                errors.append(data[metric]['std'])\n        else: # Standard train\n            if metric in data:\n                models.append(model_name)\n                scores.append(data[metric])\n                # If confidence interval exists\n                if 'confidence_intervals' in data and metric in data['confidence_intervals']:\n                    ci = data['confidence_intervals'][metric]\n                    errors.append(ci['ci_upper'] - ci['mean'])\n                else:\n                    errors.append(0)\n                    \n    if not models:\n        print(f"No data found for metric: {metric}")\n        return\n        \n    plt.figure(figsize=(10, 6))\n    x_pos = np.arange(len(models))\n    plt.bar(x_pos, scores, yerr=errors, align='center', alpha=0.7, ecolor='black', capsize=10)\n    plt.xticks(x_pos, models, rotation=45)\n    plt.ylabel(metric.capitalize())\n    plt.title(f'Model Comparison: {metric.capitalize()}')\n    plt.tight_layout()\n    plt.show()\n\nplot_model_comparison(results, 'accuracy')\nplot_model_comparison(results, 'f1_macro')

## 3. Plot Confusion Matrix

In [ ]:
def plot_confusion_matrix(data, model_name, class_names=None):\n    if 'confusion_matrix' not in data:\n        print(f"No confusion matrix for {model_name}")\n        return\n        \n    cm = np.array(data['confusion_matrix'])\n    if class_names is None:\n        class_names = [f"Class {i}" for i in range(cm.shape[0])]\n        \n    plt.figure(figsize=(8, 6))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)\n    plt.title(f'Confusion Matrix: {model_name}')\n    plt.ylabel('True Label')\n    plt.xlabel('Predicted Label')\n    plt.show()\n\nclass_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']\nfor filename, res in results.items():\n    if 'n_folds' not in res['data']:  # Only plot for standard runs for simplicity\n        plot_confusion_matrix(res['data'], res['model'], class_names)

## 4. Plot ROC Curves

In [ ]:
def plot_roc_curves(data, model_name):\n    if 'roc_curves' not in data:\n        print(f"No ROC curve data for {model_name}")\n        return\n        \n    plt.figure(figsize=(8, 6))\n    for class_name, curve in data['roc_curves'].items():\n        plt.plot(curve['fpr'], curve['tpr'], lw=2, label=class_name)\n        \n    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')\n    plt.xlim([0.0, 1.0])\n    plt.ylim([0.0, 1.05])\n    plt.xlabel('False Positive Rate')\n    plt.ylabel('True Positive Rate')\n    plt.title(f'ROC Curves: {model_name}')\n    plt.legend(loc="lower right")\n    plt.show()\n\nfor filename, res in results.items():\n    if 'n_folds' not in res['data']:\n        plot_roc_curves(res['data'], res['model'])